In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
dataset_path = os.path.join(BASE_DIR, "hand_gesture_dataset_processed")

In [ ]:
train_dir = os.path.join(dataset_path, "train")
val_dir = os.path.join(dataset_path, "val")
test_dir = os.path.join(dataset_path, "test")

In [ ]:
train_datagen = ImageDataGenerator()
val_datagen = ImageDataGenerator()
test_datagen = ImageDataGenerator()

In [ ]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    class_mode='categorical',
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    class_mode='categorical',
    shuffle=False
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    class_mode='categorical',
    shuffle=False
)

Found 534 images belonging to 8 classes.
Found 153 images belonging to 8 classes.
Found 153 images belonging to 8 classes.
Found 77 images belonging to 8 classes.


In [ ]:
num_classes = len(train_data.class_indices)

In [ ]:
with open(os.path.join(BASE_DIR, "class_indices.json"), "w") as f:
    json.dump(train_data.class_indices, f)

In [ ]:
model = models.Sequential([
    # 1. Input Layer + First Convolution
    # Using BatchNormalization after pool for stable training
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(50, 50, 1)),
    layers.MaxPooling2D((2, 2)),

    # 2. Second Convolution (extracts more complex features)
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # 3. Third Convolution
    layers.Conv2D(64, (3, 3), activation='relu'),

    # 4. Flattening (converts 2D features into a 1D vector for the classifier)
    layers.Flatten(),

    # 5. Dense Layers (The Classifier)
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.6),

    # 6. Output Layer
    # Use 'softmax' for multi-class classification
    layers.Dense(num_classes, activation='softmax')
])

c:\Users\louis\Documents\Telerobotics\computer_vision_speed_control\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 22, 22, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 11, 11, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 9, 9, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 5184)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       331,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 388,104 (1.48 MB)

 Trainable params: 388,104 (1.48 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
checkpoint_path = os.path.join(BASE_DIR, "hand_gesture_model_best.keras")
checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)
earlystop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
callbacks = [checkpoint, earlystop]

In [ ]:
print("\nTraining the model...")
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)


Training the model...
Epoch 1/20
14/17 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1651 - loss: 2.0478
Epoch 1: val_loss improved from None to 1.62763, saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras

Epoch 1: finished saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.2247 - loss: 1.9652 - val_accuracy: 0.3987 - val_loss: 1.6276
Epoch 2/20
15/17 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3700 - loss: 1.6902
Epoch 2: val_loss improved from 1.62763 to 1.36630, saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras

Epoch 2: finished saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3895 - loss: 1.6352 - val_accuracy: 0.5556 - val_loss: 1.3663
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4738 - loss: 1.4274
Epoch 3: val_lo

In [ ]:
print("\nEvaluating on test data...")
test_loss, test_accuracy = model.evaluate(test_data)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Evaluating on test data...
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5714 - loss: 1.5943
Test Loss: 1.5943
Test Accuracy: 0.5714


In [ ]:
y_true = test_data.classes
preds = model.predict(test_data, verbose=1)
y_pred = preds.argmax(axis=1)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


In [ ]:
cm = confusion_matrix(y_true, y_pred)
cr = classification_report(y_true, y_pred, target_names=[k for k, v in sorted(train_data.class_indices.items(), key=lambda x: x[1])])
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", cr)
with open(os.path.join(BASE_DIR, "classification_report.txt"), "w") as f:
    f.write("Confusion Matrix:\n")
    f.write(str(cm))
    f.write("\n\nClassification Report:\n")
    f.write(cr)


Confusion Matrix:
 [[ 0  4  1  1  1  0  0  0]
 [ 2  7  0  1  0  0  0  0]
 [ 0  0  8  2  0  0  0  0]
 [ 0  3  0  7  0  0  0  0]
 [ 0  0  0  0 10  0  0  0]
 [ 0  0  1  1  2  6  0  0]
 [ 0  9  0  0  0  0  1  0]
 [ 0  2  0  1  1  0  1  5]]

Classification Report:
                       precision    recall  f1-score   support

   background_images       0.00      0.00      0.00         7
    left_hand_images       0.28      0.70      0.40        10
level_1_speed_images       0.80      0.80      0.80        10
level_2_speed_images       0.54      0.70      0.61        10
level_3_speed_images       0.71      1.00      0.83        10
level_4_speed_images       1.00      0.60      0.75        10
   right_hand_images       0.50      0.10      0.17        10
         stop_images       1.00      0.50      0.67        10

            accuracy                           0.57        77
           macro avg       0.60      0.55      0.53        77
        weighted avg       0.63      0.57      0.55   

In [ ]:
class_names = [k for k, v in sorted(train_data.class_indices.items(), key=lambda x: x[1])]
tick_marks = np.arange(len(class_names))

# Raw confusion matrix heatmap
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
thresh = cm.max() / 2.0 if cm.size else 0
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, f"{cm[i, j]:d}", ha='center',
            color='white' if cm[i, j] > thresh else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'confusion_matrix.png'))
plt.close()

# Normalized confusion matrix (rows sum to 1)
cm_norm = cm.astype('float')
row_sums = cm_norm.sum(axis=1)[:, np.newaxis]
with np.errstate(divide='ignore', invalid='ignore'):
    cm_norm = np.divide(cm_norm, row_sums)
cm_norm = np.nan_to_num(cm_norm)

plt.figure(figsize=(8, 6))
plt.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Normalized Confusion Matrix')
plt.colorbar()
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
thresh = cm_norm.max() / 2.0 if cm_norm.size else 0
for i, j in np.ndindex(cm_norm.shape):
    plt.text(j, i, f"{cm_norm[i, j]:.2f}", ha='center',
            color='white' if cm_norm[i, j] > thresh else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label (normalized)')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'confusion_matrix_normalized.png'))
plt.close()

# Plot training curves
plt.figure()
plt.plot(history.history.get('loss', []), label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.legend()
plt.title('Loss')
plt.savefig(os.path.join(BASE_DIR, 'loss_curve.png'))
plt.close()

plt.figure()
plt.plot(history.history.get('accuracy', []), label='train_acc')
plt.plot(history.history.get('val_accuracy', []), label='val_acc')
plt.legend()
plt.title('Accuracy')
plt.savefig(os.path.join(BASE_DIR, 'accuracy_curve.png'))
plt.close()

# Save the trained model
model_save_path = os.path.join(BASE_DIR, "hand_gesture_model.keras")
model.save(model_save_path)
print(f"\nModel saved to {model_save_path}")


Model saved to c:\Users\louis\Documents\Telerobotics\hand_gesture_model.keras
